# p_n-range study: does fitting only the low-p_n (CCQE-dominated) region change the result?

A reviewer asked whether the fits should use only the low-$p_n$ events, where the sample is
almost pure CCQE, instead of the full $p_n$ range, and why a MEC knob is floated if so. Our
reading is that the $p_n$ slicing already isolates CCQE *inside* the fit: a change of $F_A$ moves
the low-$p_n$ bins, while the high-$p_n$ bins mostly pin the MEC and background normalization.
`xml/lowpn_study/` tests that directly with three restricted-range fits on the production
(`count50`) $\log_{10}Q^2$ edges, generated by `python/scripts/make_lowpn_study_xmls.py`:

| variant | reco $p_n$ bins | reads as |
|---|---|---|
| `lowpn` | the count50 bins below `PN_CUT` | the pure CCQE fit the reviewer has in mind; events above the cut are dropped |
| `lowpn_ctrl` | the same plus one merged bin $[\mathrm{PN\_CUT}, 1]$ | the explicit control-bin version of our argument: normalization and MEC, no $p_n$ shape |
| `highpn` | the count50 bins above the cut | the complement: how much $F_A$ information the high-$p_n$ region carries on its own |

Every variant is compared with a **reference** fit on the full grid: by default the `count50`
variant of the binning study (same spline box), or the production fit of the same family
(`REFERENCE_KIND = 'production'`). The quantities are the binning-independent ones of notebook 16
(goodness of fit, $\delta_F(Q^2)$, best-fit shift, $r_A^2$, profile scans) plus the nuisance
pulls, which show which knobs the restricted fits can still constrain. Section 1 does not need
any fit: it shows the interaction-mode composition of the $p_n$ bins and how much each bin
responds to the axial parameters, i.e. why the high-$p_n$ bins are not pure background control.

How to read the result (criteria agreed on 2026-10-03):

* **Asimov**: $\delta_F$ of `lowpn` relative to the reference is the information cost of dropping
  the high-$p_n$ region. Below about 10 % worse, the slices do the job on their own; `lowpn_ctrl`
  recovering most of the gap confirms the control-bin picture; `highpn` several times worse
  confirms that the high-$p_n$ region adds little by itself.
* **NuWro**: the best-fit shift between `lowpn` and the reference. Within about $0.3\,\sigma$
  the full fit is not being pulled by MEC mis-modelling at high $p_n$; a larger shift means the
  high-$p_n$ modelling drives the answer and argues for `lowpn_ctrl` or for floating more of the
  MEC model.

Fits that have not finished are listed and skipped, so the notebook can be rerun while the
study is running.

In [ ]:
from pathlib import Path
import os
import re
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

# Locate ma_zexp whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(REPO / 'python' / 'scripts') not in sys.path:
    sys.path.insert(0, str(REPO / 'python' / 'scripts'))

from postfit_physical_parameters import PUBLICATION_RC, SPECS, _fa_curves, load_fit
import make_lowpn_study_xmls as lowpn

mpl.rcParams.update(PUBLICATION_RC)
SPEC = {spec.key: spec for spec in SPECS}

## Controls

`FAMILY` selects the fake data (`nuwro` or `asimov`; environment override `LOWPN_STUDY_FAMILY`).
`REFERENCE_KIND` picks the full-range reference: `'binning'` is
`binning_study_fit_results/<family>/count50` (same widened spline box as the study XMLs),
`'production'` is `<family>_fit_results` (production chains, usually longer). A reference fit
that predates the current nuisance set cannot be loaded (its chain lacks the new branches) and is
reported as missing. Leave `FIT_KEYS = None` to compare every fit with at least one finished
variant.

In [ ]:
OUTPUT_ROOT = Path(os.environ.get('AXIAL_MASS_OUTPUT_ROOT', '/nevis/hopper/data/epelaez/axial_mass'))
STUDY_ROOT = OUTPUT_ROOT / 'lowpn_study_fit_results'
FAMILY = os.environ.get('LOWPN_STUDY_FAMILY', 'nuwro')            # nuwro or asimov
REFERENCE_KIND = os.environ.get('LOWPN_STUDY_REFERENCE', 'binning')  # 'binning' (count50) or 'production'
REFERENCE_ROOT = {'binning': OUTPUT_ROOT / 'binning_study_fit_results' / FAMILY / 'count50',
                  'production': OUTPUT_ROOT / f'{FAMILY}_fit_results'}[REFERENCE_KIND]
REFERENCE_LABEL = {'binning': 'count50 (binning study)', 'production': 'production'}[REFERENCE_KIND]
XML_ROOT = REPO / 'xml' / 'lowpn_study' / FAMILY
FIT_KEYS = None                      # e.g. ('ma', 'minerva_k6', 'minerva_k6_nuisance', 'lqcd_k6')
Q2_REFERENCE = 0.50                  # GeV^2, the comparison point of notebook 6
Q2 = np.linspace(0.0, 2.0, 201)
BURN_IN, THIN = 0, 1
Q2_CHUNK = 25                        # Q^2 points evaluated at once: every chain sample is used
PROFILE_SCAN_FIT = 'minerva_k6'      # fit whose 1D profile scans are overlaid in Section 8
INPUT_FILE = Path('/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root')
FIG_DIR = REPO / 'figs' / 'lowpn_study' / FAMILY
TABLE_DIR = REPO / 'tables' / 'lowpn_study'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PN_CUT = lowpn.PN_CUT
Q2_EDGES, PN_EDGES = np.asarray(lowpn.Q2_EDGES), np.asarray(lowpn.PN_EDGES)   # the full (count50) grid
VARIANT_ORDER = list(lowpn.VARIANTS)                                            # lowpn, lowpn_ctrl, highpn
REF = 'reference'

FIT_ORDER = [
    'ma', 'ma_uniform', 'minerva_k8', 'minerva_k7',
    'minerva_k6', 'lqcd_k7', 'lqcd_k6', 'minerva_lqcd_k7', 'minerva_lqcd_k6',
    'minerva_k6_nuisance', 'lqcd_k6_nuisance', 'minerva_lqcd_k6_nuisance',
    'minerva_k6_uniform', 'minerva_k6_uniform_nuisance',
]
# Fitted-parameter counts and uniform (penalty-free) parameters, as in notebooks 3 and 16.
N_FITTED = {
    'ma': 6, 'ma_uniform': 6,
    'lqcd_k6': 2, 'lqcd_k7': 3, 'minerva_k6': 2, 'minerva_k7': 3,
    'minerva_k8': 4, 'minerva_lqcd_k6': 2, 'minerva_lqcd_k7': 3,
    'minerva_k6_uniform': 2, 'minerva_k6_nuisance': 7, 'lqcd_k6_nuisance': 7,
    'minerva_lqcd_k6_nuisance': 7, 'minerva_k6_uniform_nuisance': 7,
}
UNIFORM_PARAMETERS = {
    'ma_uniform': {'MACCQE'},
    'minerva_k6_uniform': {'FAzexpMinervaK6PCA1', 'FAzexpMinervaK6PCA2'},
    'minerva_k6_uniform_nuisance': {'FAzexpMinervaK6PCA1', 'FAzexpMinervaK6PCA2'},
}
VARIANT_COLOR = {REF: 'black', 'lowpn': '#D55E00', 'lowpn_ctrl': '#0072B2', 'highpn': '#009E73'}
print(f'family {FAMILY!r}: study at {STUDY_ROOT / FAMILY}, reference {REFERENCE_LABEL} at {REFERENCE_ROOT}')

## 1. What the $p_n$ bins contain

Per reco $p_n$ bin of the full grid, from the selected overlay MC (weighted with `net_weight`):
the interaction-mode composition (`wc_truth_nuScatType`, GENIE codes), the signal fraction, and
the relative change of the bin content for one knot step of the fitted axial parameters
($M_A$: $+0.1$ GeV from the `MaCCQE_UBGenie` vector; MINERvA $k_{\max}=6$: $+1\,\sigma$ of PCA1).
The high-$p_n$ bins are MEC-rich but still respond to the axial parameters through their QE
content, at roughly half the low-$p_n$ rate; that is why they are not pure background control.
The table is cached in `tables/lowpn_study/pn_composition.csv`.

In [ ]:
GENIE_MODES = {1: 'QE', 10: 'MEC', 4: 'RES', 3: 'DIS', 5: 'COH'}
AXIAL_KNOBS = {'M_A +0.1 GeV': 'MaCCQE_UBGenie', 'MINERvA k6 PCA1 +1 sigma': 'weight_spline_FAzexpMinervaK6PCA1'}
COMPOSITION_FILE = TABLE_DIR / 'pn_composition.csv'


def pn_composition():
    import awkward as ak
    tree = uproot.open(INPUT_FILE)['tree']
    knobs = {label: branch for label, branch in AXIAL_KNOBS.items() if branch in tree.keys()}
    arrays = tree.arrays(['isdata', 'isext', 'isdirt', 'isnuwro', 'afro_1mu1p_sel', 'afro_1mu1p_true',
                          'afro_1mu1p_Pn', 'net_weight', 'wc_truth_nuScatType', *knobs.values()])
    overlay = ((arrays['isdata'] == 0) & (arrays['isext'] == 0) & (arrays['isdirt'] == 0)
               & (arrays['isnuwro'] == 0) & (arrays['afro_1mu1p_sel'] == 1))
    arrays = arrays[overlay]
    pn = ak.to_numpy(arrays['afro_1mu1p_Pn']); w = ak.to_numpy(arrays['net_weight'])
    mode = ak.to_numpy(arrays['wc_truth_nuScatType']); signal = ak.to_numpy(arrays['afro_1mu1p_true']) == 1
    vectors = {label: ak.to_numpy(ak.fill_none(ak.pad_none(arrays[branch], 7), 1.0)) for label, branch in knobs.items()}
    rows = []
    for lo, hi in zip(PN_EDGES[:-1], PN_EDGES[1:]):
        in_bin = (pn >= lo) & (pn < hi); total = w[in_bin].sum()
        row = {'p_n bin': f'[{lo:g}, {hi:g}]', 'events': int(in_bin.sum()), 'weighted': total}
        for code_, name in GENIE_MODES.items():
            row[f'{name} [%]'] = 100 * w[in_bin & (mode == code_)].sum() / total
        row['other [%]'] = 100 * w[in_bin & ~np.isin(mode, list(GENIE_MODES))].sum() / total
        row['signal [%]'] = 100 * w[in_bin & signal].sum() / total
        for label, vec in vectors.items():   # 7 knots, index 3 = tuned CV, one index = one knot step
            row[f'dN/N {label} [%]'] = 100 * ((w[in_bin] * vec[in_bin, 4] / vec[in_bin, 3]).sum() / total - 1)
        rows.append(row)
    return pd.DataFrame(rows)


if COMPOSITION_FILE.is_file():
    COMPOSITION = pd.read_csv(COMPOSITION_FILE)
else:
    COMPOSITION = pn_composition()
    COMPOSITION.to_csv(COMPOSITION_FILE, index=False)
display(COMPOSITION.style.format({c: '{:.1f}' for c in COMPOSITION.columns if '[%]' in c} | {'weighted': '{:.0f}'}))

fig, (ax_mode, ax_resp) = plt.subplots(1, 2, figsize=(11, 3.8))
centers, widths = 0.5 * (PN_EDGES[:-1] + PN_EDGES[1:]), np.diff(PN_EDGES)
bottom = np.zeros(len(centers))
for name, color in zip(['QE', 'MEC', 'RES', 'DIS', 'COH', 'other'], ['#56B4E9', '#E69F00', '#009E73', '#CC79A7', '#999999', '#444444']):
    values = COMPOSITION[f'{name} [%]'].to_numpy()
    ax_mode.bar(centers, values, width=widths, bottom=bottom, color=color, edgecolor='white', lw=0.5, label=name)
    bottom += values
ax_mode.plot(centers, COMPOSITION['signal [%]'], 'k.-', lw=1, label='signal (true 1mu1p)')
ax_mode.axvline(PN_CUT, color='red', lw=1, ls='--')
ax_mode.set_xlabel(r'reco $p_n$ [GeV]'); ax_mode.set_ylabel('fraction of the bin [%]'); ax_mode.set_ylim(0, 100)
ax_mode.legend(fontsize=7, ncol=2, loc='lower right')
for column in [c for c in COMPOSITION.columns if c.startswith('dN/N')]:
    ax_resp.stairs(COMPOSITION[column], PN_EDGES, baseline=None, lw=1.6, label=column.removeprefix('dN/N ').removesuffix(' [%]'))
ax_resp.axvline(PN_CUT, color='red', lw=1, ls='--', label=f'PN_CUT = {PN_CUT:g} GeV')
ax_resp.axhline(0, color='grey', lw=0.8)
ax_resp.set_xlabel(r'reco $p_n$ [GeV]'); ax_resp.set_ylabel('relative change of the bin content [%]'); ax_resp.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG_DIR.parent / 'pn_composition.pdf'); plt.show()

## 2. Inventory

A fit counts as finished when its `*_v1_PROfile.root` exists. The reference column shows which
fits can be loaded from the reference tree.

In [ ]:
RECO_UNIT = 'log10(Q^2 / GeV^2);p_n'


def reco_edges(variant):
    xmls = sorted((XML_ROOT / variant).glob('*.xml'))
    if not xmls:
        return None
    for block in re.findall(r'<bins2D\b.*?/>', xmls[0].read_text(), flags=re.DOTALL):
        unit = re.search(r'unit="([^"]*)"', block)
        if unit and unit.group(1) == RECO_UNIT:
            return tuple(tuple(float(v) for v in re.search(f'{attr}="([^"]*)"', block).group(1).split())
                         for attr in ('edgesx', 'edgesy'))
    return None


def fit_dir(variant, fit):
    return REFERENCE_ROOT / fit if variant == REF else STUDY_ROOT / FAMILY / variant / fit


def profile_root(variant, fit):
    matches = sorted(fit_dir(variant, fit).glob('*_v1_PROfile.root'))
    return matches[0] if len(matches) == 1 else None


VARIANTS = [v for v in VARIANT_ORDER if (XML_ROOT / v).is_dir()]
EDGES = {v: reco_edges(v) for v in VARIANTS}
EDGES[REF] = (tuple(Q2_EDGES), tuple(PN_EDGES))
NBINS = {v: (len(e[0]) - 1) * (len(e[1]) - 1) for v, e in EDGES.items()}
candidate_fits = FIT_ORDER if FIT_KEYS is None else list(FIT_KEYS)
FINISHED = {v: [fit for fit in candidate_fits if profile_root(v, fit) is not None] for v in [REF, *VARIANTS]}
FITS = [fit for fit in candidate_fits if any(fit in FINISHED[v] for v in VARIANTS)]
if not FITS:
    raise RuntimeError(f'no finished fit below {STUDY_ROOT / FAMILY}; run xml/lowpn_study/run_all.sh --family {FAMILY}')
INVENTORY = pd.DataFrame({v: {fit: ('done' if fit in FINISHED[v] else '') for fit in FITS} for v in [REF, *VARIANTS]})
INVENTORY.loc['bins'] = {v: NBINS[v] for v in [REF, *VARIANTS]}
display(INVENTORY)
for v in [REF, *VARIANTS]:
    print(f'{v:12s} {len(EDGES[v][0]) - 1:2d} x {len(EDGES[v][1]) - 1:2d}: p_n edges {" ".join(f"{e:g}" for e in EDGES[v][1])}')
VARIANT_LABEL = {v: f'{v} ({NBINS[v]} bins)' for v in VARIANTS} | {REF: f'{REFERENCE_LABEL} ({NBINS[REF]} bins)'}

## 3. Goodness of fit

Same bookkeeping as notebook 3: the pull penalty of the Gaussian-constrained parameters is
removed from PROfit's total, $N_{\rm dof}=N_{\rm bins}-N_{\rm fitted}$. The figure shows, per
fit and variant, $\chi^2_{\rm data}$ (not comparable across variants, the bin counts differ),
$\chi^2_{\rm data}/N_{\rm dof}$ (comparable), and the pull penalty $\chi^2_{\rm pull}$ (how far
the Gaussian-constrained parameters, axial and nuisance, are pulled from their priors). On NuWro
fake data a restricted fit with a lower $\chi^2/N_{\rm dof}$ than the full one means the dropped
bins were the ones the model described worst; a larger penalty means the restricted range pays
for its description with pulls instead. On Asimov fake data the data term is zero by construction
and only the penalty carries information.

In [ ]:
CHI2_PATTERN = re.compile(r'Global Best Fit chi(?:\^?2|²):\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)')
NBINS_PATTERN = re.compile(r'/\s*(\d+)\s+reco bins')
PARAM_PATTERN = re.compile(r'\|\|\s+(\w+)\s+:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)')


def read_fit_stats(log_file):
    if not log_file.is_file():
        return np.nan, np.nan, {}
    text = log_file.read_text(errors='replace')
    chi2 = CHI2_PATTERN.findall(text)
    nbins = NBINS_PATTERN.findall(text)
    final_block = text.rsplit('Global Best Fit chi^2:', 1)[-1]
    parameters = {name: float(value) for name, value in PARAM_PATTERN.findall(final_block)}
    return (float(chi2[-1]) if chi2 else np.nan), (int(nbins[-1]) if nbins else np.nan), parameters


records = []
for v in [REF, *VARIANTS]:
    for fit in FITS:
        if fit not in FINISHED[v]:
            continue
        chi2_tot, nbins_log, parameters = read_fit_stats(fit_dir(v, fit) / 'profile.log')
        penalty = sum(value ** 2 for name, value in parameters.items() if name not in UNIFORM_PARAMETERS.get(fit, set()))
        nbins = nbins_log if np.isfinite(nbins_log) else NBINS[v]
        ndof = nbins - N_FITTED[fit]
        records.append(dict(variant=v, fit=fit, nbins=nbins, ndof=ndof, chi2_tot=chi2_tot, chi2_penalty=penalty,
                            chi2_data=chi2_tot - penalty, chi2_data_per_ndof=(chi2_tot - penalty) / ndof))
CHI2 = pd.DataFrame(records)
chi2_table = CHI2.pivot(index='fit', columns='variant', values=['chi2_data', 'chi2_data_per_ndof', 'chi2_penalty'])
chi2_table = chi2_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['chi2_data', 'chi2_data_per_ndof', 'chi2_penalty'], [REF, *VARIANTS]]))
display(chi2_table.style.format('{:.3f}', na_rep=''))

# Figure: the three chi^2 numbers per fit, one marker per variant (reference in black).
CHI2_PANELS = [('chi2_data', r'$\chi^2_{\rm data}$'), ('chi2_data_per_ndof', r'$\chi^2_{\rm data}/N_{\rm dof}$'),
               ('chi2_penalty', r'$\chi^2_{\rm pull}$ (penalty of the constrained parameters)')]
order = [REF, *VARIANTS]
fig, axes = plt.subplots(1, len(CHI2_PANELS), figsize=(4.8 * len(CHI2_PANELS), 0.32 * len(FITS) + 2.2), sharey=True)
y = np.arange(len(FITS))[::-1]
for k, v in enumerate(order):
    part = CHI2[CHI2.variant == v].set_index('fit').reindex(FITS)
    offset = (k - (len(order) - 1) / 2) * 0.18
    for ax, (column, _) in zip(axes, CHI2_PANELS):
        ax.plot(part[column], y + offset, 'o', ms=5, color=VARIANT_COLOR[v], label=VARIANT_LABEL[v] if ax is axes[0] else None)
for ax, (column, label) in zip(axes, CHI2_PANELS):
    ax.set_xlabel(label); ax.grid(axis='x', alpha=0.3); ax.set_axisbelow(True)
axes[1].axvline(1, color='grey', lw=0.8)
axes[0].set_yticks(y, FITS, fontsize=8); axes[0].legend(fontsize=8)
fig.suptitle(f'{FAMILY}: goodness of fit per p_n range against {REFERENCE_LABEL}', fontsize=10)
fig.tight_layout(); fig.savefig(FIG_DIR / 'chi2.pdf'); plt.show()

## 4. Posterior chains

`load_fit` transforms every chain to physical parameters and, for the fits with nuisances,
keeps the nuisance pulls next to them. A reference fit whose chain predates the current
nuisance set raises a `KeyError` on the missing branch and is skipped with a message.

In [ ]:
RESULTS = {v: {} for v in [REF, *VARIANTS]}
for v in [REF, *VARIANTS]:
    for fit in FITS:
        if fit not in FINISHED[v]:
            continue
        # load_fit reads <data_root>/<suite>/<fit>/: the reference tree and the study variants
        root, suite = (REFERENCE_ROOT.parent, REFERENCE_ROOT.name) if v == REF else (STUDY_ROOT / FAMILY, v)
        try:
            result = load_fit(SPEC[fit], suite=suite, burn_in=BURN_IN, thin=THIN, data_root=root)
        except KeyError as error:
            print(f'{v:12s} {fit:28s} skipped: chain lacks {error} (fit predates the current nuisance set?)')
            continue
        if result is None:
            continue
        RESULTS[v][fit] = result
        print(f'{v:12s} {fit:28s} {len(result["samples"]):>8,} samples')
for fit in FITS:
    lengths = {v: len(RESULTS[v][fit]['samples']) for v in [REF, *VARIANTS] if fit in RESULTS[v]}
    if len(set(lengths.values())) > 1:
        print(f'NOTE {fit}: chain lengths differ, ' + ', '.join(f'{v}={n:,}' for v, n in lengths.items())
              + ' (widths and medians stay comparable, the shorter chains carry more MCMC noise)')

## 5. $F_A(Q^2)$: precision and shift

$\delta_F(Q^2)=\sigma[F_A]/|\mathbb{E}[F_A]|$ over every chain sample (definition of notebook 6).
The headline is the ratio $\delta_F^{\rm variant}/\delta_F^{\rm ref}$ at $Q^2_{\rm ref}$: the
information cost (or gain) of the restricted range. The mean shift is
$(\mathbb{E}[F_A]_{\rm variant}-\mathbb{E}[F_A]_{\rm ref})/\sigma[F_A]_{\rm ref}$. The second
figure shows $\delta_F(Q^2)$ and the posterior $-F_A$ band per fit.

In [ ]:
def fa_band(result, q2, use_prior=False, quantiles=(16, 50, 84)):
    q2 = np.atleast_1d(np.asarray(q2, dtype=float))
    n = len(result['prior_samples' if use_prior else 'samples'])
    mean, sigma = np.empty(len(q2)), np.empty(len(q2))
    pct = np.empty((len(quantiles), len(q2)))
    for start in range(0, len(q2), Q2_CHUNK):
        stop = min(start + Q2_CHUNK, len(q2))
        points = q2[start:stop] if stop - start > 1 else np.r_[q2[start:stop], q2[start:stop] + 1e-6]
        curves = _fa_curves(result, points, use_prior=use_prior, max_samples=n)[:, :stop - start]
        mean[start:stop], sigma[start:stop] = curves.mean(axis=0), curves.std(axis=0)
        pct[:, start:stop] = np.percentile(curves, quantiles, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        frac = sigma / np.abs(mean)
    return dict(mean=mean, sigma=sigma, frac=frac, q16=pct[0], median=pct[1], q84=pct[2])


rows = []
BANDS = {v: {} for v in [REF, *VARIANTS]}
IREF = int(np.argmin(np.abs(Q2 - Q2_REFERENCE)))
IN_RANGE = (Q2 > 0.05) & (Q2 <= 1.0)
for fit in FITS:
    for v in [REF, *VARIANTS]:
        if fit in RESULTS[v]:
            BANDS[v][fit] = fa_band(RESULTS[v][fit], Q2)
    ref = BANDS[REF].get(fit)
    if ref is not None:
        rows.append(dict(fit=fit, variant=REF, nbins=NBINS[REF], **{'posterior [%]': 100 * ref['frac'][IREF],
                         'ratio to reference': 1.0, 'mean shift [sigma_ref]': 0.0, 'max ratio Q2<=1': 1.0}))
    for v in VARIANTS:
        band = BANDS[v].get(fit)
        if band is None:
            continue
        rows.append(dict(
            fit=fit, variant=v, nbins=NBINS[v],
            **{'posterior [%]': 100 * band['frac'][IREF],
               'ratio to reference': band['frac'][IREF] / ref['frac'][IREF] if ref is not None else np.nan,
               'mean shift [sigma_ref]': (band['mean'][IREF] - ref['mean'][IREF]) / ref['sigma'][IREF] if ref is not None else np.nan,
               'max ratio Q2<=1': np.nanmax(band['frac'][IN_RANGE] / ref['frac'][IN_RANGE]) if ref is not None else np.nan}))
FA = pd.DataFrame(rows)
fa_table = FA.pivot(index='fit', columns='variant', values=['posterior [%]', 'ratio to reference', 'mean shift [sigma_ref]'])
fa_table = fa_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['posterior [%]', 'ratio to reference', 'mean shift [sigma_ref]'], [REF, *VARIANTS]]))
print(f'delta_F at Q^2 = {Q2_REFERENCE} GeV^2, reference = {REFERENCE_LABEL}')
display(fa_table.style.format('{:.3f}', na_rep=''))

# Headline: one marker per fit and variant.
fig, axes = plt.subplots(1, 2, figsize=(12, 0.32 * len(FITS) + 2.2), sharey=True)
y = np.arange(len(FITS))[::-1]
for k, v in enumerate(VARIANTS):
    part = FA[FA.variant == v].set_index('fit').reindex(FITS)
    offset = (k - (len(VARIANTS) - 1) / 2) * 0.22
    axes[0].plot(part['ratio to reference'], y + offset, 'o', color=VARIANT_COLOR[v], label=VARIANT_LABEL[v])
    axes[1].plot(part['mean shift [sigma_ref]'], y + offset, 'o', color=VARIANT_COLOR[v])
axes[0].axvline(1, color='grey', lw=0.8); axes[0].axvspan(0.9, 1.1, color='#f1f3f5', zorder=0)
axes[1].axvline(0, color='grey', lw=0.8); axes[1].axvspan(-0.3, 0.3, color='#f1f3f5', zorder=0)
axes[0].set_xlabel(rf'$\delta_F(Q^2={Q2_REFERENCE})$ / reference'); axes[0].set_xscale('log')
axes[1].set_xlabel(rf'shift of $\mathbb{{E}}[F_A]$ at $Q^2={Q2_REFERENCE}$  [$\sigma_{{\rm ref}}$]')
axes[0].set_yticks(y, FITS, fontsize=8); axes[0].legend(fontsize=8)
for ax in axes:
    ax.grid(axis='x', alpha=0.3, which='both')
fig.suptitle(f'{FAMILY}: restricted p_n ranges against {REFERENCE_LABEL}', fontsize=10)
fig.tight_layout(); fig.savefig(FIG_DIR / 'headline_deltaF_shift.pdf'); plt.show()

In [ ]:
for fit in FITS:
    available = [v for v in [REF, *VARIANTS] if fit in BANDS[v]]
    if not available:
        continue
    fig = plt.figure(figsize=(14, 5))
    grid = fig.add_gridspec(2, 2, height_ratios=[3, 1.2], hspace=0.05, wspace=0.25)
    ax_frac = fig.add_subplot(grid[:, 0]); ax_fa = fig.add_subplot(grid[0, 1]); ax_ratio = fig.add_subplot(grid[1, 1], sharex=ax_fa)
    ref_median = BANDS[REF][fit]['median'] if fit in BANDS[REF] else None
    for v in available:
        band = BANDS[v][fit]
        ax_frac.plot(Q2, 100 * band['frac'], color=VARIANT_COLOR[v], lw=1.8 if v == REF else 1.3, label=VARIANT_LABEL[v])
        ax_fa.plot(Q2, -band['median'], color=VARIANT_COLOR[v], lw=1.3)
        ax_fa.fill_between(Q2, -band['q84'], -band['q16'], color=VARIANT_COLOR[v], alpha=0.15, lw=0)
        if ref_median is not None:
            with np.errstate(divide='ignore', invalid='ignore'):
                ax_ratio.plot(Q2, band['median'] / ref_median, color=VARIANT_COLOR[v], lw=1.3)
                ax_ratio.fill_between(Q2, band['q16'] / ref_median, band['q84'] / ref_median, color=VARIANT_COLOR[v], alpha=0.12, lw=0)
    prior = fa_band(RESULTS[available[0]][fit], Q2, use_prior=True)
    ax_frac.plot(Q2, 100 * prior['frac'], color='grey', ls='--', lw=1.2, label='prior')
    ax_frac.axvline(Q2_REFERENCE, color='grey', lw=0.8, ls=':')
    ax_frac.set_xlabel(r'$Q^2$ [GeV$^2$]'); ax_frac.set_ylabel(r'$\delta_F(Q^2)$ [%]'); ax_frac.legend(fontsize=8)
    ax_frac.set_ylim(0, min(ax_frac.get_ylim()[1], 60))
    ax_fa.set_ylabel(r'$-F_A(Q^2)$'); ax_fa.tick_params(labelbottom=False)
    ax_ratio.axhline(1, color='grey', lw=0.8); ax_ratio.set_ylabel('median / ref'); ax_ratio.set_xlabel(r'$Q^2$ [GeV$^2$]')
    fig.suptitle(f'{FAMILY}: {fit}', y=0.98, fontsize=10)
    fig.savefig(FIG_DIR / f'fa_band_{fit}.pdf', bbox_inches='tight'); plt.show()

## 6. Fitted parameters and nuisance pulls

Top: the independent axial parameters ($M_A$, or $a_1\ldots a_{k_{\max}-4}$) per variant, with
the half-width relative to the reference and the shift of the median in units of the reference
half-width. Bottom, for the fits that float nuisances: the posterior median of each pull and its
68 % half-width divided by the half-width of its prior (drawn with the same truncation as the
fit's `restrict`). A ratio near 1 means the data do not constrain the knob; in `lowpn` that is
expected for the MEC knobs (`XSecShape_CCMEC`, `DecayAngMEC`), which is the quantitative answer
to "why float a MEC knob if there are no MEC events".

In [ ]:
def joint_summary(result):
    summary = result['summary'].iloc[result['joint_indices']]
    return pd.DataFrame({'median': summary['posterior_median'], 'half_width': 0.5 * (summary['q84'] - summary['q16'])})


def nuisance_pulls(result):
    # (names, posterior samples, prior samples) of the fitted nuisance pulls, or None
    if 'ma_joint_samples' in result:
        names, post, prior = result['ma_joint_names'][1:], result['ma_joint_samples'][:, 1:], result['ma_joint_prior_samples'][:, 1:]
    elif 'zexp_nuisance_samples' in result:
        n_axial = len(result['joint_indices'])
        names, post, prior = result['zexp_nuisance_names'][n_axial:], result['zexp_nuisance_samples'][:, n_axial:], result['zexp_nuisance_prior_samples'][:, n_axial:]
    else:
        return None
    return list(names), post, prior


rows, pull_rows = [], []
for fit in FITS:
    ref = joint_summary(RESULTS[REF][fit]) if fit in RESULTS[REF] else None
    for v in [REF, *VARIANTS]:
        result = RESULTS[v].get(fit)
        if result is None:
            continue
        summary = joint_summary(result)
        for name, row in summary.iterrows():
            rows.append(dict(fit=fit, parameter=name, variant=v, median=row['median'], half_width=row['half_width'],
                             width_ratio=row['half_width'] / ref.loc[name, 'half_width'] if ref is not None else np.nan,
                             shift_sigma=(row['median'] - ref.loc[name, 'median']) / ref.loc[name, 'half_width'] if ref is not None else np.nan))
        pulls = nuisance_pulls(result)
        if pulls is None:
            continue
        names, post, prior = pulls
        q16, q50, q84 = np.percentile(post, [16, 50, 84], axis=0)
        p16, p84 = np.percentile(prior, [16, 84], axis=0)
        for i, name in enumerate(names):
            pull_rows.append(dict(fit=fit, knob=name.removesuffix(' pull'), variant=v, median=q50[i],
                                  half_width=0.5 * (q84[i] - q16[i]), width_over_prior=(q84[i] - q16[i]) / (p84[i] - p16[i])))
PARAMS = pd.DataFrame(rows)
param_table = PARAMS.pivot(index=['fit', 'parameter'], columns='variant', values=['median', 'half_width', 'width_ratio', 'shift_sigma'])
param_table = param_table.reindex(columns=pd.MultiIndex.from_product([['median', 'half_width', 'width_ratio', 'shift_sigma'], [REF, *VARIANTS]]))
display(param_table.style.format('{:.3f}', na_rep=''))

PULLS = pd.DataFrame(pull_rows)
if len(PULLS):
    pull_table = PULLS.pivot(index=['fit', 'knob'], columns='variant', values=['median', 'width_over_prior'])
    pull_table = pull_table.reindex(columns=pd.MultiIndex.from_product([['median', 'width_over_prior'], [REF, *VARIANTS]]))
    print('nuisance pulls: posterior median, and 68% width over the prior width (1 = unconstrained by the data)')
    display(pull_table.style.format('{:.2f}', na_rep=''))

    fits_with_pulls = [fit for fit in FITS if fit in set(PULLS.fit)]
    knobs = list(dict.fromkeys(PULLS.knob))
    fig, axes = plt.subplots(1, len(fits_with_pulls), figsize=(3.4 * len(fits_with_pulls), 0.45 * len(knobs) + 1.8), sharey=True, squeeze=False)
    y = np.arange(len(knobs))[::-1]
    for ax, fit in zip(axes[0], fits_with_pulls):
        for k, v in enumerate([REF, *VARIANTS]):
            part = PULLS[(PULLS.fit == fit) & (PULLS.variant == v)].set_index('knob').reindex(knobs)
            if part['median'].isna().all():
                continue
            offset = (k - len(VARIANTS) / 2) * 0.18
            ax.errorbar(part['median'], y + offset, xerr=part['half_width'], fmt='o', ms=3.5, color=VARIANT_COLOR[v], capsize=2, lw=1, label=VARIANT_LABEL[v])
        ax.axvline(0, color='grey', lw=0.8); ax.set_title(fit, fontsize=9); ax.set_xlabel('pull'); ax.grid(axis='x', alpha=0.3)
    axes[0][0].set_yticks(y, knobs, fontsize=8); axes[0][0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'nuisance_pulls.pdf'); plt.show()

## 7. Axial radius $r_A^2$

Definitions of notebook 7: $r_A^2 = 12/M_A^2$ for the dipole fits, the analytic slope of the
$z$-expansion at $Q^2=0$ otherwise, in fm$^2$; with the fraction of samples at $r_A^2<0$.

In [ ]:
HBARC_GEV_FM = 0.1973269804


def dz_dq2_at_zero(t0, t_cut):
    a, b = np.sqrt(t_cut), np.sqrt(t_cut - t0)
    return b / (a * (a + b) ** 2)


def radius_squared(result, use_prior=False):
    samples = result['prior_samples' if use_prior else 'samples']
    prior = result['spec'].prior
    if prior is None:
        return 12.0 / np.asarray(samples[:, 0], dtype=float) ** 2 * HBARC_GEV_FM ** 2
    coefficients = np.atleast_2d(np.asarray(samples, dtype=float))
    t0, t_cut = prior.t0_gev2, prior.t_cut_gev2
    z0 = (np.sqrt(t_cut) - np.sqrt(t_cut - t0)) / (np.sqrt(t_cut) + np.sqrt(t_cut - t0))
    k = np.arange(1, coefficients.shape[1])
    dfa_dq2 = (coefficients[:, 1:] @ (k * z0 ** (k - 1))) * dz_dq2_at_zero(t0, t_cut)
    return -6.0 / prior.fa_q2_zero * dfa_dq2 * HBARC_GEV_FM ** 2


rows = []
for fit in FITS:
    ref = radius_squared(RESULTS[REF][fit]) if fit in RESULTS[REF] else None
    ref_q = np.percentile(ref, [16, 50, 84]) if ref is not None else None
    for v in [REF, *VARIANTS]:
        if fit not in RESULTS[v]:
            continue
        r2 = radius_squared(RESULTS[v][fit])
        q16, med, q84 = np.percentile(r2, [16, 50, 84])
        rows.append(dict(fit=fit, variant=v, **{'median [fm^2]': med, 'half_width': 0.5 * (q84 - q16),
                         'width_ratio': (q84 - q16) / (ref_q[2] - ref_q[0]) if ref_q is not None else np.nan,
                         'shift_sigma': (med - ref_q[1]) / (0.5 * (ref_q[2] - ref_q[0])) if ref_q is not None else np.nan,
                         'negative [%]': 100 * np.mean(r2 < 0)}))
RADIUS = pd.DataFrame(rows)
radius_table = RADIUS.pivot(index='fit', columns='variant', values=['median [fm^2]', 'width_ratio', 'shift_sigma', 'negative [%]'])
radius_table = radius_table.reindex(index=FITS).reindex(columns=pd.MultiIndex.from_product(
    [['median [fm^2]', 'width_ratio', 'shift_sigma', 'negative [%]'], [REF, *VARIANTS]]))
display(radius_table.style.format('{:.3f}', na_rep=''))

## 8. Profile scans of one fit

PROfit's 1D $\Delta\chi^2$ scans per fitted parameter (fit coordinates), overlaid across the
variants for `PROFILE_SCAN_FIT`. Wider parabolas in `lowpn` are the lost information; a displaced
minimum means the restricted range prefers a different best fit.

In [ ]:
def read_profile_points(variant, fit):
    matches = sorted(fit_dir(variant, fit).glob('*_v1_PROfile_points.txt'))
    if not matches:
        return None
    rows = []
    for line in matches[0].read_text().splitlines():
        if not line or line.startswith('#'):
            continue
        index, kind, name, value, dchi2, chi2 = line.split()
        rows.append(dict(index=int(index), kind=kind, parameter=name, value=float(value), dchi2=float(dchi2), chi2=float(chi2)))
    return pd.DataFrame(rows)


scans = {v: read_profile_points(v, PROFILE_SCAN_FIT) for v in [REF, *VARIANTS] if PROFILE_SCAN_FIT in FINISHED[v]}
scans = {v: s for v, s in scans.items() if s is not None and not s.empty}
if not scans:
    print(f'no profile scans found for {PROFILE_SCAN_FIT!r}')
else:
    parameters = list(dict.fromkeys(p for s in scans.values() for p in s.parameter))
    fig, axes = plt.subplots(1, len(parameters), figsize=(4.2 * len(parameters), 3.6), squeeze=False)
    for ax, parameter in zip(axes[0], parameters):
        for v, scan in scans.items():
            part = scan[scan.parameter == parameter].sort_values('value')
            ax.plot(part.value, part.dchi2, 'o-', ms=3, color=VARIANT_COLOR[v], lw=1.8 if v == REF else 1.2, label=VARIANT_LABEL[v])
        ax.axhline(1, color='grey', lw=0.8, ls=':'); ax.axhline(4, color='grey', lw=0.8, ls=':')
        ax.set_xlabel(parameter); ax.set_ylabel(r'$\Delta\chi^2$'); ax.set_ylim(0, min(ax.get_ylim()[1], 30))
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f'{FAMILY}: {PROFILE_SCAN_FIT} profile scans per p_n range', y=1.02)
    fig.tight_layout(); fig.savefig(FIG_DIR / f'profile_scans_{PROFILE_SCAN_FIT}.pdf', bbox_inches='tight'); plt.show()

## 9. Summary

One row per fit and variant with the numbers the criteria above refer to, written to
`tables/lowpn_study/summary_<family>.csv`. Flags: `precision_ok` is a $\delta_F$ ratio below
1.1 at $Q^2_{\rm ref}$, `shift_ok` a mean shift below $0.3\,\sigma_{\rm ref}$.

In [ ]:
SUMMARY = (FA[FA.variant != REF]
           .merge(CHI2[['variant', 'fit', 'chi2_data_per_ndof']], on=['variant', 'fit'], how='left')
           .merge(RADIUS[['variant', 'fit', 'shift_sigma', 'negative [%]']].rename(columns={'shift_sigma': 'r2 shift_sigma', 'negative [%]': 'r2 negative [%]'}),
                  on=['variant', 'fit'], how='left'))
# Flags stay empty where the reference fit is missing.
SUMMARY['precision_ok'] = (SUMMARY['ratio to reference'] < 1.1).where(SUMMARY['ratio to reference'].notna())
SUMMARY['shift_ok'] = (SUMMARY['mean shift [sigma_ref]'].abs() < 0.3).where(SUMMARY['mean shift [sigma_ref]'].notna())
SUMMARY.insert(0, 'family', FAMILY)
SUMMARY.insert(1, 'reference', REFERENCE_LABEL)
SUMMARY = SUMMARY.set_index(['fit', 'variant']).reindex(pd.MultiIndex.from_product([FITS, VARIANTS], names=['fit', 'variant'])).dropna(how='all')
display(SUMMARY.style.format({c: '{:.3f}' for c in SUMMARY.columns if SUMMARY[c].dtype.kind == 'f'}, na_rep=''))
SUMMARY.to_csv(TABLE_DIR / f'summary_{FAMILY}.csv', float_format='%.5g')
print(f'wrote {(TABLE_DIR / f"summary_{FAMILY}.csv").relative_to(REPO)}')